# Handling Message Blocks

### Calling Claude with the JSON Schema
Helping it understand it has a tool available

In [1]:
from dotenv import load_dotenv
from utils.util_funcs import *

load_dotenv()
client, model = api_client_setup()

In [2]:
get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Return the current local datetime from the runtime environment as a formatted string. Use this tool when the user asks for the current date, time, or a current datetime in a custom format. Do not use it for date arithmetic, future/past calculations, or timezone conversion. The optional date_format parameter must be a non-empty Python strftime format string. If omitted, the tool uses '%Y-%m-%d %H:%M:%S'.",
  "strict": True,
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "minLength": 1,
        "description": "Optional Python strftime format string used to format the current datetime. Must not be empty. Defaults to '%Y-%m-%d %H:%M:%S'. Example: '%A, %B %d, %Y %I:%M:%S %p'.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "additionalProperties": False
  },
  "input_examples": [
    {},
    {
      "date_format": "%A, %B %d, %Y %I:%M:%S %p"
    }
  ]
})

In [10]:
messages = []

messages.append({ 
    "role": "user",
    "content": "What is the exact time, formatted in HH:MM:SS?"
})
messages

[{'role': 'user', 'content': 'What is the exact time, formatted in HH:MM:SS?'}]

In [11]:
response = client.messages.create(
    model = model,
    max_tokens = 1000, 
    messages = messages, 
    tools = [get_current_datetime_schema]
)

In [12]:
messages.append({ 
    "role": "assistant",
    "content": response.content
})

There is now a new object called a `ToolUseBlock`, in addition to the `TextBlock` object

In [13]:
messages

[{'role': 'user', 'content': 'What is the exact time, formatted in HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='EpECCokBCA8YAipAfWq+NtMvU9QOr+el7cAFfTiDxRx69YhrDlKuAZn+yeUSnmZ6a01LTIdX/yV/CXb6bQgE9d0jBeZfxpSGhLxhPTIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRjNjMxY2E0MS0wZjQ2LTRkOWEtODFiYS0wNjJhNmYxZDNjY2ESDG0Vv5jxksvHx2DISBoMqYG5y37YcV7TOHIgIjAl/FiPUYuvA9uCc9GGfDHOSH0rFoKWqWeogvGV4FmArdjDpzJsxYJD4rR+tLHluxcqNXdh8gFaEwMEOuTK0gk+cHAl3yGI0aekGwAUtp60Rb7m4Cv2M1uzLiCCntjYk/8xBNVTh778GAE=', thinking='', type='thinking'),
   ToolUseBlock(id='toolu_011L4rtP6pi3Z7dx1aeicjME', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]